# ForenSight — AI-Image Detector (Transfer Learning)

This notebook fine-tunes a **pretrained MobileNetV2** to tell **real photos**
from **AI-generated images** (dataset: CIFAKE). You run the cells top-to-bottom.
The trained model file is then plugged into ForenSight as its ML fallback.

**How to use:** Runtime menu -> *Change runtime type* -> Hardware accelerator ->
**GPU**. Then run each cell in order (Shift+Enter).


## 1. Check the GPU is on
If `GPU available` prints an empty list, turn it on via Runtime -> Change runtime type -> GPU.


In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Download the dataset (CIFAKE, from Kaggle)
Get your Kaggle API key first: kaggle.com -> your profile -> *Settings* -> *Create New API Token*.
That downloads a small file **kaggle.json**. Run the cell, then upload that file when asked.


In [ ]:
from google.colab import files
print('Upload your kaggle.json (Kaggle -> Settings -> Create New API Token)')
files.upload()  # choose kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
!unzip -q -o cifake-real-and-ai-generated-synthetic-images.zip -d cifake
print('Done. Folders:')
!ls cifake

## 3. Load the images
CIFAKE is already split into `train/` and `test/`, each with `REAL/` and `FAKE/` sub-folders.
Keras reads the label from the folder name. We resize everything to 128x128.


In [ ]:
IMG_SIZE = (128, 128)
BATCH = 32
train_ds = tf.keras.utils.image_dataset_from_directory(
    'cifake/train', image_size=IMG_SIZE, batch_size=BATCH,
    label_mode='binary', shuffle=True, seed=42)
test_ds = tf.keras.utils.image_dataset_from_directory(
    'cifake/test', image_size=IMG_SIZE, batch_size=BATCH,
    label_mode='binary', shuffle=False)
class_names = train_ds.class_names
print('Classes (index order):', class_names)   # ['FAKE', 'REAL'] -> FAKE=0, REAL=1
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)   # stream from disk (no .cache -> low RAM)
test_ds = test_ds.prefetch(AUTOTUNE)

## 4. Peek at a few images (sanity check)


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i][0])])
        plt.axis('off')
plt.show()

## 5. Build the model (transfer learning)
We load **MobileNetV2** already trained on millions of images, **freeze** it (keep its
learned 'vision'), and add a small new **head** that outputs one number: the probability
the image is REAL. `Rescaling` scales pixels to the [-1, 1] range MobileNetV2 expects.
The two `Random*` layers gently augment the data during training only.


In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False   # freeze the pretrained backbone

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.05)(x)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)   # == MobileNetV2 preprocessing
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)   # 1 number: P(REAL)
model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Train
An *epoch* = one full pass over the training images. 5 epochs is plenty here (a few minutes
on the GPU). Watch `val_accuracy` — that's accuracy on images it did **not** learn from.


In [ ]:
EPOCHS = 5
history = model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS)

## 7. Training curves
If train accuracy keeps rising but val accuracy stalls/drops, that's *overfitting*.


In [ ]:
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.title('Accuracy'); plt.show()

## 8. Evaluate — precision, recall, confusion matrix
These are the numbers for your Results chapter.


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
y_true, y_prob = [], []
for images, labels in test_ds:
    y_true.extend(labels.numpy().flatten().tolist())
    y_prob.extend(model.predict(images, verbose=0).flatten().tolist())
y_true = np.array(y_true)
y_pred = (np.array(y_prob) >= 0.5).astype(int)
print('Confusion matrix [rows=true, cols=pred]  order:', class_names)
print(confusion_matrix(y_true, y_pred))
print()
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## 9. Save the model + download it
This saves `cifake_ai_detector.keras` and downloads it to your laptop. You'll drop this file
into ForenSight (`forensight/models/`).


In [ ]:
model.save('cifake_ai_detector.keras')
from google.colab import files
files.download('cifake_ai_detector.keras')

## 10. Try it on your own image
Upload any picture; the model prints how likely it thinks the image is AI-generated.
(Remember: the model outputs P(REAL); P(AI) = 1 - P(REAL).)


In [ ]:
from google.colab import files
up = files.upload()
for fn in up:
    img = tf.keras.utils.load_img(fn, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)[None, ...]
    p_real = float(model.predict(arr, verbose=0)[0][0])
    print(f'{fn}:  P(AI) = {(1 - p_real):.1%}   P(real) = {p_real:.1%}')

## 11. (Optional) Fine-tune for a small accuracy boost
Unfreeze the top layers of the backbone and train a little more with a **tiny** learning rate.
Skip this the first time; come back if you want higher accuracy.


In [ ]:
base.trainable = True
for layer in base.layers[:-30]:   # keep all but the last 30 layers frozen
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=test_ds, epochs=3)
model.save('cifake_ai_detector.keras')

---
### What next
1. **Phase 2:** re-run this notebook on a higher-resolution realistic dataset
   (Kaggle *AI vs Human-Generated Images* or HF *AI-vs-Real*) at `IMG_SIZE=(224,224)`
   so it works on real photos, not just tiny 32x32 ones.
2. **Phase 3:** send me the `.keras` file and I'll write `ml_detector.py` to plug it
   into ForenSight as the ML fallback (a new 'AI (ML): NN%' signal).
3. **Phase 4:** evaluate on your own `gambar/` images and report precision/recall vs the
   rule-based provenance detector — your standout Results chapter.
